In [1]:
from datasets import load_dataset
import pandas as pd

stream = load_dataset(
    "phreshphish/phreshphish",
    split="train",
    streaming=True,
    columns=["url", "label"],
)

sample = stream.shuffle(
    seed=42,
    buffer_size=10_000,
).take(30_000)

df_phresh = pd.DataFrame(list(sample))

display(df_phresh.head())
print(df_phresh["label"].value_counts())

/home/pmtfedora/Documents/skole/Dat158/PhishingURL/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,url,label
0,https://en.wikipedia.org/wiki/Sangiovese,benign
1,https://www.xvideos.com/models/thick634,benign
2,www.boursedirect.fr/fr/marche/new-york,benign
3,https://gouchevlaw.com/3-must-documents-llc/,benign
4,https://www.almanac.com/plant/crocuses,benign


label
benign    16662
phish     13338
Name: count, dtype: int64


In [2]:
from urllib.parse import urlsplit

print("Missing values:")
print(df_phresh.isna().sum())

print("\nDuplicate URLs:", df_phresh["url"].duplicated().sum())

has_www = df_phresh["url"].apply(
    lambda url: (
        urlsplit(url if "://" in url else "https://" + url).hostname or ""
    ).lower().startswith("www.")
)

display(pd.crosstab(
    df_phresh["label"],
    has_www.rename("Has www"),
    normalize="index",
).mul(100).round(2))

Missing values:
url      0
label    0
dtype: int64

Duplicate URLs: 0


Has www,False,True
label,,
benign,41.20,58.80
phish,96.07,3.93


In [4]:
from urllib.parse import urlsplit

def url_to_model_text(url):
    url = url.strip()
    parsed = urlsplit(url if "://" in url else "https://" + url)

    hostname = (parsed.hostname or "").lower()

    if hostname.startswith("www."):
        hostname = hostname[4:]

    text = hostname + parsed.path

    if parsed.query:
        text += "?" + parsed.query

    if parsed.fragment:
        text += "#" + parsed.fragment

    return text

In [5]:
from sklearn.model_selection import train_test_split

texts = df_phresh["url"].apply(url_to_model_text)
y = df_phresh["label"].map({"phish": 0, "benign": 1})

assert y.notna().all(), "Unexpected labels found"

texts_train, texts_val, y_train, y_val = train_test_split(
    texts,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

majority_class = y_train.mode().iloc[0]
print("Validation baseline:", (y_val == majority_class).mean())

Validation baseline: 0.5553333333333333


In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

phresh_model = make_pipeline(
    TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=3,
        max_features=100_000,
        lowercase=True,
    ),
    SGDClassifier(
        loss="log_loss",
        random_state=42,
        max_iter=1000,
    ),
)

phresh_model.fit(texts_train, y_train)
predictions = phresh_model.predict(texts_val)

print("Validation accuracy:", accuracy_score(y_val, predictions))
print("Phishing precision:", precision_score(
    y_val, predictions, pos_label=0, zero_division=0
))
print("Phishing recall:", recall_score(
    y_val, predictions, pos_label=0, zero_division=0
))
print("Phishing F1:", f1_score(
    y_val, predictions, pos_label=0, zero_division=0
))

Validation accuracy: 0.9376666666666666
Phishing precision: 0.9803182579564489
Phishing recall: 0.8774362818590704
Phishing F1: 0.9260284810126582


In [8]:
import pandas as pd

example_urls = pd.Series([
    # Legitimate websites
    "https://www.google.com",
    "https://www.microsoft.com",
    "https://www.apple.com",
    "https://www.amazon.com",
    "https://www.netflix.com",
    "https://www.paypal.com",
    "https://www.github.com",
    "https://www.spotify.com",
    "https://www.reddit.com",
    "https://www.wikipedia.org",
    "https://www.dnb.no",
    "https://www.vg.no",
    "https://www.hvl.no",
    "https://login.microsoftonline.com",
    "https://accounts.google.com",

    # Typosquatting / lookalike domains
    "https://www.g00gle.com",
    "https://www.netf1ix.com",
    "https://www.paypa1.com",
    "https://www.amaz0n.com",
    "https://www.micros0ft.com",
    "https://www.app1e.com",
    "https://www.faceb00k.com",
    "https://www.sp0tify.com",
    "https://www.githüb.com",

    # Fake login pages
    "http://paypal-login.example.com",
    "https://microsoft-account-verify.example.com",
    "http://netflix-billing-update.example.com",
    "https://appleid-security-alert.example.com",
    "http://google-account-recovery.example.com",
    "https://dnb-bankid-login.example.com",

    # Suspicious subdomains
    "https://paypal.com.security-check.example.com",
    "https://login.microsoft.com.verify.example.com",
    "https://accounts.google.com.auth.example.com",
    "https://secure.amazon.com.payment.example.com",

    # Suspicious URL structures
    "http://192.168.1.100/login",
    "http://203.0.113.42/verify/account",
    "https://example.com/login?session=123456",
    "https://example.com/secure/update-password",
    "http://example.com/verify?email=user@test.com",
    "https://example.com/account/confirm-identity",

    # Legitimate URLs with suspicious-looking paths
    "https://github.com/login",
    "https://accounts.google.com/signin",
    "https://www.paypal.com/signin",
    "https://www.microsoft.com/en-us/security",
    "https://www.amazon.com/gp/help/customer/display.html",
    "https://www.dnb.no/privat/nettbank.html",

    # Shorteners / redirects
    "https://bit.ly/3abcXYZ",
    "https://tinyurl.com/example123",
    "https://t.co/abc123",
])

example_predictions = phresh_model.predict(
    example_urls.apply(url_to_model_text)
)

display(pd.DataFrame({
    "URL": example_urls,
    "Prediction": [
        "Phishing" if prediction == 0 else "Legitimate"
        for prediction in example_predictions
    ],
}))

,URL,Prediction
0,https://www.google.com,Phishing
1,https://www.microsoft.com,Legitimate
2,https://www.apple.com,Legitimate
3,https://www.amazon.com,Legitimate
4,https://www.netflix.com,Phishing
5,https://www.paypal.com,Phishing
6,https://www.github.com,Phishing
7,https://www.spotify.com,Phishing
8,https://www.reddit.com,Legitimate
9,https://www.wikipedia.org,Legitimate


In [10]:
print(df_phresh["label"].value_counts())
print(phresh_model.classes_)

label
benign    16662
phish     13338
Name: count, dtype: int64
[0 1]


In [11]:
print(df_phresh[["label"]].head(10))
print(y_train.value_counts())

    label
0  benign
1  benign
2  benign
3  benign
4  benign
5  benign
6  benign
7  benign
8  benign
9   phish
label
1    13330
0    10670
Name: count, dtype: int64


In [12]:
print("Labels in the dataset:")
print(df_phresh["label"].value_counts())

print("\nClasses learned by the model:")
print(phresh_model.classes_)

# Compare existing training labels with the original dataset
label_check = pd.DataFrame({
    "Original label": df_phresh.loc[y_train.index, "label"],
    "Training label": y_train,
})

display(pd.crosstab(
    label_check["Original label"],
    label_check["Training label"],
))

expected_labels = label_check["Original label"].map({
    "phish": 0,
    "benign": 1,
})

assert expected_labels.notna().all(), "Unexpected dataset labels"
assert expected_labels.eq(y_train).all(), "Label mapping does not match"

print("\nMapping confirmed: 0 = phishing, 1 = legitimate")

Labels in the dataset:
label
benign    16662
phish     13338
Name: count, dtype: int64

Classes learned by the model:
[0 1]


Training label,0,1
Original label,,
benign,0,13330
phish,10670,0



Mapping confirmed: 0 = phishing, 1 = legitimate


In [13]:
example_urls = pd.Series([
    "https://netflix.com",
    "https://netf1ix.com",
    "https://www.microsoft.com",
    "http://rnicrosoft.com",
    "https://paypal.com",
    "https://paypai.com",
    "https://amazon.com",
    "https://amaz0n.com",
])

example_texts = example_urls.apply(url_to_model_text)

predictions = phresh_model.predict(example_texts)
probabilities = phresh_model.predict_proba(example_texts)

# Find the probability column for class 0 explicitly
phishing_column = list(phresh_model.classes_).index(0)

display(pd.DataFrame({
    "URL": example_urls,
    "Model input": example_texts,
    "Predicted class": predictions,
    "Prediction": [
        "Phishing" if prediction == 0 else "Legitimate"
        for prediction in predictions
    ],
    "Phishing score": probabilities[:, phishing_column],
}))

,URL,Model input,Predicted class,Prediction,Phishing score
0,https://netflix.com,netflix.com,0,Phishing,0.581971
1,https://netf1ix.com,netf1ix.com,1,Legitimate,0.448480
2,https://www.microsoft.com,microsoft.com,1,Legitimate,0.241079
3,http://rnicrosoft.com,rnicrosoft.com,1,Legitimate,0.247674
4,https://paypal.com,paypal.com,0,Phishing,0.637497
5,https://paypai.com,paypai.com,0,Phishing,0.532518
6,https://amazon.com,amazon.com,1,Legitimate,0.186901
7,https://amaz0n.com,amaz0n.com,1,Legitimate,0.257277


In [15]:
import ipaddress
from urllib.parse import urlsplit


def extract_structural_features(url):
    parsed = urlsplit(
        url.strip() if "://" in url else "https://" + url.strip()
    )
    hostname = (parsed.hostname or "").lower()

    if hostname.startswith("www."):
        hostname = hostname[4:]

    try:
        ipaddress.ip_address(hostname)
        is_ip = 1
    except ValueError:
        is_ip = 0

    host_digits = sum(character.isdigit() for character in hostname)

    # Count transitions between letters and digits
    letter_digit_transitions = sum(
        (left.isalpha() and right.isdigit())
        or (left.isdigit() and right.isalpha())
        for left, right in zip(hostname, hostname[1:])
    )

    text = hostname + parsed.path
    if parsed.query:
        text += "?" + parsed.query

    return {
        "text_length": len(text),
        "hostname_length": len(hostname),
        "path_length": len(parsed.path),
        "query_length": len(parsed.query),
        "hostname_dots": hostname.count("."),
        "hostname_hyphens": hostname.count("-"),
        "hostname_digits": host_digits,
        "hostname_digit_ratio": host_digits / max(len(hostname), 1),
        "letter_digit_transitions": letter_digit_transitions,
        "is_ip": is_ip,
        "has_userinfo": int(parsed.username is not None),
        "percent_signs": text.count("%"),
        "equals_signs": text.count("="),
        "ampersands": text.count("&"),
        "path_slashes": parsed.path.count("/"),
        "has_punycode": int(
            any(part.startswith("xn--") for part in hostname.split("."))
        ),
    }


X_structural = pd.DataFrame(
    [
        extract_structural_features(url)
        for url in df_phresh["url"]
    ],
    index=df_phresh.index,
)

X_train_structural = X_structural.loc[y_train.index]
X_val_structural = X_structural.loc[y_val.index]

display(X_structural.head())

,text_length,hostname_length,path_length,query_length,hostname_dots,hostname_hyphens,hostname_digits,hostname_digit_ratio,letter_digit_transitions,is_ip,has_userinfo,percent_signs,equals_signs,ampersands,path_slashes,has_punycode
0,32,16,16,0,2,0,0,0.0,0,0,0,0,0,0,2,0
1,27,11,16,0,1,0,0,0.0,0,0,0,0,0,0,2,0
2,34,15,19,0,1,0,0,0.0,0,0,0,0,0,0,3,0
3,36,14,22,0,1,0,0,0.0,0,0,0,0,0,0,2,0
4,26,11,15,0,1,0,0,0.0,0,0,0,0,0,0,2,0


In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

rf_structural = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

rf_structural.fit(X_train_structural, y_train)
rf_val_predictions = rf_structural.predict(X_val_structural)

print("Training accuracy:", rf_structural.score(
    X_train_structural, y_train
))
print("Validation accuracy:", accuracy_score(
    y_val, rf_val_predictions
))
print("Phishing precision:", precision_score(
    y_val, rf_val_predictions, pos_label=0, zero_division=0
))
print("Phishing recall:", recall_score(
    y_val, rf_val_predictions, pos_label=0, zero_division=0
))
print("Phishing F1:", f1_score(
    y_val, rf_val_predictions, pos_label=0, zero_division=0
))

Training accuracy: 0.9265833333333333
Validation accuracy: 0.892
Phishing precision: 0.8992094861660079
Phishing recall: 0.8526986506746627
Phishing F1: 0.8753366679492113


In [17]:
X_examples = pd.DataFrame([
    extract_structural_features(url)
    for url in example_urls
])

example_predictions_rf = rf_structural.predict(X_examples)

display(pd.DataFrame({
    "URL": example_urls,
    "Prediction": [
        "Phishing" if prediction == 0 else "Legitimate"
        for prediction in example_predictions_rf
    ],
}))

,URL,Prediction
0,https://netflix.com,Phishing
1,https://netf1ix.com,Phishing
2,https://www.microsoft.com,Phishing
3,http://rnicrosoft.com,Phishing
4,https://paypal.com,Phishing
5,https://paypai.com,Phishing
6,https://amazon.com,Phishing
7,https://amaz0n.com,Phishing


In [18]:
url_shapes = pd.DataFrame({
    "label": df_phresh["label"],
    "path_length": X_structural["path_length"],
    "query_length": X_structural["query_length"],
})

url_shapes["homepage_like"] = (
    (url_shapes["path_length"] <= 1)
    & (url_shapes["query_length"] == 0)
)

display(pd.crosstab(
    url_shapes["label"],
    url_shapes["homepage_like"],
    normalize="index",
).mul(100).round(2))

homepage_like,False,True
label,,
benign,93.03,6.97
phish,55.47,44.53


In [19]:
validation_results = pd.DataFrame({
    "actual": y_val,
    "predicted": phresh_model.predict(texts_val),
    "homepage_like": url_shapes.loc[y_val.index, "homepage_like"],
})

rows = []

for homepage_like, group in validation_results.groupby("homepage_like"):
    benign = group["actual"] == 1

    rows.append({
        "URL group": "Homepage-like" if homepage_like else "Other URLs",
        "Examples": len(group),
        "Benign examples": int(benign.sum()),
        "Accuracy": accuracy_score(group["actual"], group["predicted"]),
        "Phishing recall": recall_score(
            group["actual"], group["predicted"],
            pos_label=0, zero_division=0,
        ),
        "Benign false alarm rate": (
            (group.loc[benign, "predicted"] == 0).mean()
            if benign.any() else float("nan")
        ),
    })

display(pd.DataFrame(rows).round(4))

,URL group,Examples,Benign examples,Accuracy,Phishing recall,Benign false alarm rate
0,Other URLs,4598,3109,0.9548,0.8805,0.0096
1,Homepage-like,1402,223,0.8816,0.8736,0.0762


In [20]:
false_alarm_indices = validation_results.index[
    validation_results["homepage_like"]
    & (validation_results["actual"] == 1)
    & (validation_results["predicted"] == 0)
]

display(df_phresh.loc[false_alarm_indices, ["url", "label"]])

,url,label
15445,https://shop.webtoon.com/,benign
20786,https://www.stmaryhudson.cc/,benign
5064,https://eforms.cso.ie/,benign
14541,https://www.paixdieubeer.be/,benign
6378,www.tf1info.fr/,benign
23129,https://lunchbox.io/,benign
24933,https://poczta.domena.pl/,benign
10943,https://champagne.pages.dev/,benign
19383,https://www.teamvaal.co.za/,benign
5277,https://login.kramp.com/,benign


In [22]:
import numpy as np
phishing_column = list(phresh_model.classes_).index(0)
phishing_scores = phresh_model.predict_proba(texts_val)[:, phishing_column]

benign_mask = y_val.to_numpy() == 1
homepage_mask = validation_results.loc[
    y_val.index, "homepage_like"
].to_numpy()

rows = []

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    predictions = np.where(phishing_scores >= threshold, 0, 1)
    benign_homepages = benign_mask & homepage_mask

    rows.append({
        "Threshold": threshold,
        "Phishing precision": precision_score(
            y_val, predictions, pos_label=0, zero_division=0
        ),
        "Phishing recall": recall_score(
            y_val, predictions, pos_label=0, zero_division=0
        ),
        "Phishing F1": f1_score(
            y_val, predictions, pos_label=0, zero_division=0
        ),
        "Benign false alarm rate": (
            predictions[benign_mask] == 0
        ).mean(),
        "Homepage false alarm rate": (
            (predictions[benign_homepages] == 0).mean()
            if benign_homepages.any() else float("nan")
        ),
    })

display(pd.DataFrame(rows).round(4))

,Threshold,Phishing precision,Phishing recall,Phishing F1,Benign false alarm rate,Homepage false alarm rate
0,0.3,0.8804,0.9771,0.9263,0.1062,0.4664
1,0.4,0.9508,0.9337,0.9421,0.0387,0.1883
2,0.5,0.9803,0.8774,0.9260,0.0141,0.0762
3,0.6,0.9918,0.8178,0.8965,0.0054,0.0314
4,0.7,0.9975,0.7384,0.8486,0.0015,0.0090
5,0.8,0.9994,0.6312,0.7737,0.0003,0.0045
6,0.9,0.9991,0.4370,0.6081,0.0003,0.0045


In [23]:
from datasets import load_dataset

test_stream = load_dataset(
    "phreshphish/phreshphish",
    split="test",
    streaming=True,
    columns=["url", "label"],
)

df_test = pd.DataFrame(list(
    test_stream.shuffle(seed=42, buffer_size=10_000).take(10_000)
))

test_texts = df_test["url"].apply(url_to_model_text)
test_labels = df_test["label"].map({"phish": 0, "benign": 1})

assert test_labels.notna().all(), "Unexpected test labels"

test_scores = phresh_model.predict_proba(
    test_texts
)[:, phishing_column]

test_predictions = np.where(test_scores >= 0.5, 0, 1)

print("Test examples:", len(df_test))
print("Test accuracy:", accuracy_score(test_labels, test_predictions))
print("Phishing precision:", precision_score(
    test_labels, test_predictions, pos_label=0, zero_division=0
))
print("Phishing recall:", recall_score(
    test_labels, test_predictions, pos_label=0, zero_division=0
))
print("Phishing F1:", f1_score(
    test_labels, test_predictions, pos_label=0, zero_division=0
))

Test examples: 10000
Test accuracy: 0.9004
Phishing precision: 0.9794940079893475
Phishing recall: 0.8000870132695236
Phishing F1: 0.8807471264367817


In [24]:
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)

joblib.dump(
    phresh_model,
    "../models/phresh_text_model.joblib",
)

['../models/phresh_text_model.joblib']